# Scraping Quiz Test Data
This Notebook is for scraping quiz test data from various online sources to be used in the Driving Quizzes Project.


## 1. alfanak.net Website
http://code.alfanak.net/

In [ ]:
import requests
from bs4 import BeautifulSoup
import os
import json



In [15]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

url = "https://code.alfanak.net/"
html_page = requests.get(url,headers=headers,verify=False)
html_page_soup = BeautifulSoup(html_page.content, 'html.parser')
print(html_page_soup.prettify())

/home/abdelhak/miniconda3/envs/dl/lib/python3.12/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'code.alfanak.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


<html>
 <body>
  <script src="/aes.js" type="text/javascript">
  </script>
  <script>
   function toNumbers(d){var e=[];d.replace(/(..)/g,function(d){e.push(parseInt(d,16))});return e}function toHex(){for(var d=[],d=1==arguments.length&&arguments[0].constructor==Array?arguments[0]:arguments,e="",f=0;f<d.length;f++)e+=(16>d[f]?"0":"")+d[f].toString(16);return e.toLowerCase()}var a=toNumbers("f655ba9d09a112d4968c63579db590b4"),b=toNumbers("98344c2eee86c3994890592585b49f80"),c=toNumbers("f30ccbcd9bd2f8f6f434cbba817dbf17");document.cookie="__test="+toHex(slowAES.decrypt(c,2,a,b))+"; max-age=21600; expires=Thu, 31-Dec-37 23:55:55 GMT; path=/"; location.href="https://code.alfanak.net/?i=1";
  </script>
  <noscript>
   This site requires Javascript to work, please enable Javascript in your browser or use a browser with Javascript support
  </noscript>
 </body>
</html>



In [ ]:
# pip install selenium webdriver-manager
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

opts = Options()
opts.headless = True
driver = webdriver.Chrome( options=opts)
driver.get("http://code.alfanak.net/?t=1")
# wait a second for redirect/JS
cookie = driver.get_cookie("__test")
print(cookie)
driver.quit()


{'domain': 'code.alfanak.net', 'expiry': 1760992445, 'httpOnly': False, 'name': '__test', 'path': '/', 'sameSite': 'Lax', 'secure': False, 'value': '4812acfa3f0c862ad7e053edf5bf39ca'}


In [39]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time

# --- Setup Selenium ---
chrome_options = Options()
chrome_options.add_argument("--headless")  # remove this if you want to watch the browser
chrome_options.add_argument("--disable-gpu")

driver = webdriver.Chrome( options=chrome_options)

# --- Variables ---
url = "http://code.alfanak.net"
book_pages = []

# --- Scrape pages 1 to 20 ---
for page_number in range(1, 21):
    page_url = f"{url}/index.php?t={page_number}&i=1"
    driver.get(page_url)

    # Give time for page and JS/cookies to load
    time.sleep(1)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    book_pages.append(soup)

    print(f"Scraped page {page_number}")

# --- Finish ---
driver.quit()
print("✅ All pages scraped successfully!")


Scraped page 1
Scraped page 2
Scraped page 3
Scraped page 4
Scraped page 5
Scraped page 6
Scraped page 7
Scraped page 8
Scraped page 9
Scraped page 10
Scraped page 11
Scraped page 12
Scraped page 13
Scraped page 14
Scraped page 15
Scraped page 16
Scraped page 17
Scraped page 18
Scraped page 19
Scraped page 20
✅ All pages scraped successfully!


In [65]:
## each page has three sections to be scraped 
## h1 class="title" and text الإشارات
## h1 class="title" and text أولوية المرور
## h1 class="title" and text الأسئلة
## image of the book page  will be scraped later easiy using `url/images/test-01.jpg`
## lets start by extracting pages sections
sections_names =['الإشارات','أولوية المرور','الأسئلة']
def get_section(section_name,page_html):
    h1_tags = page_html.find_all('h1', class_='title')
    for h1 in h1_tags:
        if h1.get_text(strip=True) == section_name:
            # Get the section content after the h1 tag
            section_content = h1.find_next()
            return section_content
    return None
# test 
section = get_section(section_name=sections_names[0],page_html=book_pages[0])
print(section)


<div class="row1"><br/>
1- حذار، خطر غير معين.<br/>
2- الدوران إلى اليمين ممنوع.<br/>
3- التوقف ممنوع من 16 لـ 31 من كل شهر.<br/>
4- التزام السير باتجاه اليمين إجباري.<br/>
5- نهاية كل الممنوعات ماعدا الوقوف و التوقف.<br/>
6- حذار، تقاطع طرق (الأولوية لليمين).<br/>
7- نهاية منع التجاوز بالنسبة للشاحنات.<br/>
8- موقف سيارات الأجرة.<br/>
9- نهاية الممر الإجباري للدراجات.<br/>
10- الدخول إلى منطقة التوقف فيها ممنوع.<br/>
11- ممر إجباري للدراجات.<br/>
12- الخروج من منطقة التوقف فيها ممنوع.<br/>
13- غابة سريعة الاشتعال.<br/>
14- مرور الدرجات والدراجات النارية ممنوع.<br/>
15- حذار، طريق ذو اتجاهين.<br/>
16- الدخول إلى منطقة يمنع فيها تجاوز السرعة 30كم/سا.<br/>
<br/></div>


In [ ]:
## scrape all sections from all pages
scraped_data = {}
for section_name in sections_names:
    scraped_data[section_name] = []
    for page_html in book_pages:
        section_content = get_section(section_name, page_html)
        scraped_data[section_name].append(section_content)
# save scraped data to files
output_dir = "../data/scraped_quiz_test_data"
os.makedirs(output_dir, exist_ok=True)
file_name = os.path.join(output_dir, "scraped_data.json")
with open(file_name, "w", encoding="utf-8") as f:
    json.dump({k: [str(v) for v in vals] for k, vals in scraped_data.items()}, f, ensure_ascii=False, indent=4)
print(f"✅ Scraped data saved to {file_name}")

✅ Scraped data saved to ../data/scraped_quiz_test_data/scraped_data.json


In [ ]:
import re

def clean_text(text):
    # clean any html tags and what is betweeen them
    clean_text = re.sub(r'<[^>]+>', '', text)
    return clean_text

In [59]:
print(f"Section 1 : Example {clean_text( scraped_data['الإشارات'][0])}")
print(f"Section 2 : Example {clean_text(scraped_data['أولوية المرور'][0])}")
print(f"Section 3 : Example {clean_text(scraped_data['الأسئلة'][0])}")

Section 1 : Example <div class="row1"><br/>
1- حذار، خطر غير معين.<br/>
2- الدوران إلى اليمين ممنوع.<br/>
3- التوقف ممنوع من 16 لـ 31 من كل شهر.<br/>
4- التزام السير باتجاه اليمين إجباري.<br/>
5- نهاية كل الممنوعات ماعدا الوقوف و التوقف.<br/>
6- حذار، تقاطع طرق (الأولوية لليمين).<br/>
7- نهاية منع التجاوز بالنسبة للشاحنات.<br/>
8- موقف سيارات الأجرة.<br/>
9- نهاية الممر الإجباري للدراجات.<br/>
10- الدخول إلى منطقة التوقف فيها ممنوع.<br/>
11- ممر إجباري للدراجات.<br/>
12- الخروج من منطقة التوقف فيها ممنوع.<br/>
13- غابة سريعة الاشتعال.<br/>
14- مرور الدرجات والدراجات النارية ممنوع.<br/>
15- حذار، طريق ذو اتجاهين.<br/>
16- الدخول إلى منطقة يمنع فيها تجاوز السرعة 30كم/سا.<br/>
<br/></div>
Section 2 : Example <div class="row1"><p>1- محور دوراني مع إشارة ترك المرور - تمر السيارة <strong style="color:#b51a1a">الحمراء</strong> ثم <strong style="color:#CFC01F">الصفراء</strong>.</p><p>2- تقاطع طرق مع إشارة قف على 150 متر و إشارة قف - تمر السيارتان <strong style="color:#0a3da6">الزرقاء</strong> 

In [66]:
def parse_section_1(section_div):
    lines = section_div.decode_contents().split("<br/>")
    items = []
    for line in lines:
        text = line.strip().replace("<br>", "")
        if text and text[0].isdigit():
            items.append(text)
    return items
def parse_section_2(section_div):
    items = []
    for p in section_div.find_all("p"):
        text = p.get_text(" ", strip=True)
        items.append(text)
    return items
def parse_section_3(section_div):
    lines = section_div.decode_contents().split("<br/>")
    items = []
    current_block = []

    for line in lines:
        text = line.strip().replace("<br>", "")
        if not text:
            continue
        if text[0].isdigit() and "-" in text[:3]:  # new numbered item
            if current_block:
                items.append("\n".join(current_block))
                current_block = []
        current_block.append(text)

    if current_block:
        items.append("\n".join(current_block))

    return items
def parse_section(section_name,section):
    if section_name =="الإشارات":
        return parse_section_1(section)
    elif section_name =="أولوية المرور":
        return parse_section_2(section)
    elif section_name=="الأسئلة":
        return parse_section_3(section)
    else:
        raise ValueError("Invalid Section Name")


In [ ]:
section_1_items = parse_section_1(scraped_data['الإشارات'][1])
section_2_items = parse_section_2(scraped_data["أولوية المرور"][1])
section_3_items = parse_section_3(scraped_data["الأسئلة"][1])

print("\n".join(section_1_items))
print("\n".join(section_2_items))
print("\n".join(section_3_items))



1- السرعة الأدنى الإجبارية 30 كلم/سا.
2- نهاية منع تجاوز السرعة 50 كلم/سا.
3- طريق ذات أولوية.
4- حذار، محور دوراني.
5- نهاية أولويّة الطريق.
6- الاتجاه إلى اليمين إجباري.
7- ترك المرور على بعد 150م.
8- الدوران إلى اليمين ممنوع.
9- حذار، تعاقب عدة منعرجات أولها إلى اليمين.
10- حذار، ممهل في نفس مكان الإشارة.
11- مرور الشاحنات ممنوع.
12- اتجاه ممنوع.
13- تجاوز السرعة 30 ممنوع.
14- التجاوز ممنوع.
15- نهاية منع التجاوز.
16- التجاوز ممنوع بالنسبة للشاحنات.
1- تقاطع طرق مع إشارة حذار محور دوراني وإشارة ترك المرور - تمر السيارة الزرقاء ثم البيضاء .
2- تقاطع طرق مع إشارة حذار محور دوراني وإشارة ترك المرور - تمر السيارتان الخضراء و الصفراء في نفس الوقت، ثم تمر الحمراء .
3- تقاطع طرق مع إشارة طريق ذات أولوية - تمر السيارتان الصفراء و الزرقاء ، ثم تمر الحمراء .
4- تقاطع طرق مع إشارة ترك المرور - تمر السيارتان الزرقاء و الصفراء ، ثم الحمراء .
5- تقاطع طرق مع إشارة ترك المرور - تمر السيارتان الحمراء و الزرقاء ، ثم السيارة الصفراء .
6- تقاطع طرق مع إشارة نهاية طريق ذات أولوية على بعد 400م و إشارة ت

In [70]:
# now for all sections and all pages  from scraped_data
all_parsed_data = {}
for section_name in sections_names:
    sections_data = scraped_data[section_name]
    all_parsed_data[section_name]= []
    for s in sections_data:
        s_parsed = parse_section(section_name,s)
        all_parsed_data[section_name].append(clean_text(s_parsed))
# save all parsed data to file parsed_quiz_test_data.json
output_dir = "../data/parsed_quiz_test_data"
os.makedirs(output_dir, exist_ok=True)
file_name = os.path.join(output_dir, "parsed_quiz_test_data.json")
with open(file_name, "w", encoding="utf-8") as f:
    json.dump(all_parsed_data, f, ensure_ascii=False, indent=4)
print(f"✅ Parsed data saved to {file_name}")


✅ Parsed data saved to ../data/parsed_quiz_test_data/parsed_quiz_test_data.json


SyntaxError: incomplete input (1808518401.py, line 2)